In [12]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
from torchvision import models
from tqdm import tqdm

In [13]:
# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔹 Using device: {device}")

# 🔹 Load datasets
train_dir = "D:/Projects/Minor Project/Deepfake Detection/datasets/New folder/real-vs-fake/train"  # Update with correct path
val_dir = "D:/Projects/Minor Project/Deepfake Detection/datasets/New folder/real-vs-fake/valid"  # Update with correct path

🔹 Using device: cpu


In [14]:
# Define faster transformations
transform = transforms.Compose([
    transforms.Resize((160, 160)),  # Reduce image size for faster processing
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [15]:
# Load datasets
try:
    train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
    val_dataset = datasets.ImageFolder(root=val_dir, transform=transform)
    print(f"✅ Train Dataset: {len(train_dataset)} images")
    print(f"✅ Validation Dataset: {len(val_dataset)} images")
except Exception as e:
    print(f"❌ Error loading datasets: {e}")

✅ Train Dataset: 100000 images
✅ Validation Dataset: 20000 images


In [16]:
# Load data (Assuming train_dataset and val_dataset are already defined)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

In [17]:
# Load EfficientNet-B3 model
from torchvision import models

model = models.efficientnet_b3(pretrained=True)

In [18]:
# Modify the classifier for binary classification (Real vs Fake)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 2)
model = model.to(device)

In [19]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.0005)  # Faster convergence with a slightly higher LR

# Enable mixed precision
scaler = torch.cuda.amp.GradScaler()    

In [20]:
# Training loop
num_epochs = 3  # Reduce epochs for quick execution
best_acc = 0.0  

for epoch in range(num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for inputs, labels in progress_bar:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        # Enable mixed precision training
        with torch.cuda.amp.autocast():
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        # Backpropagation with GradScaler
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

        progress_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{(correct/total)*100:.2f}%")

Epoch 1/3:   0%|          | 0/6250 [00:00<?, ?it/s]

c:\Users\Divyaa Sriram\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\amp\autocast_mode.py:250: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
Epoch 2/3:  32%|███▏      | 2016/6250 [1:20:18<2:48:38,  2.39s/it, acc=72.42%, loss=0.5096]


KeyboardInterrupt: 

In [ ]:
    # Calculate epoch loss and accuracy
    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    print(f"🔹 Epoch {epoch+1}: Loss = {epoch_loss:.4f}, Accuracy = {epoch_acc:.2f}%")

    # Validation Loop
    model.eval()
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for val_inputs, val_labels in val_loader:
            val_inputs, val_labels = val_inputs.to(device), val_labels.to(device)

            val_outputs = model(val_inputs)
            _, val_preds = torch.max(val_outputs, 1)

            val_correct += torch.sum(val_preds == val_labels).item()
            val_total += val_labels.size(0)

    val_acc = (val_correct / val_total) * 100
    print(f"✅ Validation Accuracy: {val_acc:.2f}%")

In [ ]:
    # Save best model based on validation accuracy
    if val_acc > best_acc:
        best_acc = val_acc
        model_path = "best_efficient_model.pth"  # Save in Google Drive
        torch.save(model.state_dict(), model_path)
        print(f"🎉 Best model saved at epoch {epoch+1} with validation accuracy {best_acc:.2f}%")

print("✅ Training complete!")